# Data Scientist Salary Prediction

**Tabular Regression Project** — Predict data scientist salaries from job postings.

Models: CatBoost · LightGBM · XGBoost · FLAML · LazyPredict
Dataset: Glassdoor Jobs (956 rows × 15 columns)
Target: Derived salary (avg of salary range)

## 2 · Project Overview

We predict **data scientist salaries** from Glassdoor job posting features including job title, company rating, location, industry, and company size. The main challenge is **extracting a numeric salary** from the text-based `Salary Estimate` column.

With 956 rows, this is a small-to-medium dataset where careful preprocessing matters.

## 3 · Learning Objectives

1. Parse salary ranges from text fields.
2. Handle messy real-world job posting data.
3. Encode high-cardinality categorical features.
4. Predict salaries from company and job features.
5. Use LazyPredict and FLAML for benchmarking.

## 4 · Problem Statement

Given Glassdoor job posting features (title, company, location, industry, size, rating), predict the **average salary** for the position. This helps job seekers and recruiters benchmark compensation.

## 5 · Why This Project Matters

- **Salary transparency** helps reduce pay gaps.
- Job seekers can negotiate better with salary predictions.
- Companies can benchmark their offers against market rates.
- Teaches real-world data cleaning on messy text fields.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 956 |
| **Columns** | 15 |
| **Features** | Job Title, Rating, Company Name, Location, Size, Founded, Industry, Sector, Revenue, etc. |
| **Target** | Derived from Salary Estimate (avg of min/max, in $K) |
| **File** | `glassdoor_jobs.csv` (local) |

## 7 · Dataset Source and License Notes

- **Source**: Glassdoor data science job postings (scraped/collected).
- **License**: Educational / research use.
- **Note**: Data reflects a snapshot in time — salaries change rapidly.

## 8 · Environment Setup

In [ ]:
import subprocess, sys

def _install_if_missing(pkg, import_name=None):
    try:
        __import__(import_name or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg, imp in [('catboost', None), ('lightgbm', None), ('xgboost', None),
                 ('flaml', None), ('lazypredict', None)]:
    _install_if_missing(pkg, imp)

print('All packages ready.')

## 9 · Imports

In [ ]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred): return mean_squared_error(y_true, y_pred, squared=False)

warnings.filterwarnings('ignore')
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('Imports complete.')

## 10 · Configuration / Constants

In [ ]:
SEED = 42
TEST_SIZE = 0.2
np.random.seed(SEED)

## 11 · Dataset Download or Loading

In [ ]:
DATA_PATH = os.path.join(SAVE_DIR, 'glassdoor_jobs.csv')
assert os.path.exists(DATA_PATH), f'Dataset not found at {DATA_PATH}'
df = pd.read_csv(DATA_PATH)
print(f'Loaded: {df.shape}')
df.head()

## 12 · Data Validation Checks

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nDuplicates: {df.duplicated().sum()}')
print(f'\nSalary Estimate sample:')
print(df['Salary Estimate'].head(10))

## 13 · Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['Rating'].hist(bins=20, ax=axes[0], edgecolor='black')
axes[0].set_title('Company Rating Distribution')

df['Location'].value_counts().head(15).plot(kind='barh', ax=axes[1])
axes[1].set_title('Top 15 Locations')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## 14 · Target Analysis

We parse the Salary Estimate column to extract numeric salary values.

In [ ]:
import re

def parse_salary(s):
    """Extract average salary from text like '$62K-$112K (Glassdoor est.)'"""
    if pd.isna(s) or s == '-1':
        return np.nan
    # Remove non-numeric except K and -
    s = s.replace('$', '').replace(',', '').replace('(Glassdoor est.)', '').replace('(Employer est.)', '').strip()
    # Find numbers with K
    nums = re.findall(r'(\d+)K?', s)
    if len(nums) >= 2:
        return (int(nums[0]) + int(nums[1])) / 2
    elif len(nums) == 1:
        return int(nums[0])
    return np.nan

df['avg_salary'] = df['Salary Estimate'].apply(parse_salary)
df = df.dropna(subset=['avg_salary'])
TARGET = 'avg_salary'

print(f'Salary stats (in $K):')
print(df[TARGET].describe())
print(f'Skewness: {df[TARGET].skew():.2f}')

## 15 · Train / Validation / Test Split

## 16 · Preprocessing Strategy

- Drop text-heavy columns: Job Description, Competitors, Company Name.
- Parse Rating (handle -1 as missing).
- Encode remaining categoricals.
- Extract state from Location.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Drop Unnamed index, high-text columns
drop_cols = ['Unnamed: 0', 'Salary Estimate', 'Job Description', 'Competitors', 'Company Name', 'Headquarters']
for c in drop_cols:
    if c in df.columns:
        df = df.drop(columns=[c])

# Handle Rating = -1
df['Rating'] = df['Rating'].replace(-1, np.nan)
df['Rating'] = df['Rating'].fillna(df['Rating'].median())

# Extract state from Location
df['state'] = df['Location'].apply(lambda x: x.split(',')[-1].strip() if ',' in str(x) else 'Unknown')
df = df.drop(columns=['Location'])

# Handle Founded
df['Founded'] = df['Founded'].replace(-1, np.nan)
df['company_age'] = 2026 - df['Founded']
df['company_age'] = df['company_age'].fillna(df['company_age'].median())
df = df.drop(columns=['Founded'])

# Encode remaining categoricals
for col in df.select_dtypes(include='object').columns:
    if df[col].nunique() > 50:
        df = df.drop(columns=[col])
    else:
        df[col] = df[col].fillna('Unknown')
        df[col] = LabelEncoder().fit_transform(df[col])

print(f'Preprocessed: {df.shape}')
print(df.dtypes)

## 17 · Feature Engineering

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 18 · Baseline Model

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)

baseline_rmse = root_mean_squared_error(y_test, y_pred_base)
baseline_mae = mean_absolute_error(y_test, y_pred_base)
baseline_r2 = r2_score(y_test, y_pred_base)
print(f'Baseline LR: RMSE={baseline_rmse:.2f}, MAE={baseline_mae:.2f}, R²={baseline_r2:.4f}')

## 19 · LazyPredict Benchmark

In [ ]:
from lazypredict.Supervised import LazyRegressor
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print(lazy_models.head(15))

## 20 · FLAML AutoML Run

In [ ]:
try:
    from flaml import AutoML
    flaml_model = AutoML()
    flaml_model.fit(
        X_train, y_train,
        task='regression',
        time_budget=60,
        metric='rmse',
        seed=SEED,
        verbose=0
    )
    y_pred_flaml = flaml_model.predict(X_test)
    flaml_rmse = root_mean_squared_error(y_test, y_pred_flaml)
    flaml_mae = mean_absolute_error(y_test, y_pred_flaml)
    flaml_r2 = r2_score(y_test, y_pred_flaml)
    print(f'FLAML best: {flaml_model.best_estimator}')
    print(f'  RMSE: {flaml_rmse:.2f}')
    print(f'  MAE:  {flaml_mae:.2f}')
    print(f'  R²:   {flaml_r2:.4f}')
except Exception as e:
    print(f'FLAML failed: {e}')
    flaml_rmse = flaml_mae = flaml_r2 = None

## 21 · Additional Best-Library Workflow (CatBoost + LightGBM + XGBoost)

In [ ]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

models = {
    'CatBoost': CatBoostRegressor(iterations=300, learning_rate=0.1, depth=6,
                                   random_seed=SEED, verbose=0),
    'LightGBM': LGBMRegressor(n_estimators=300, learning_rate=0.1, max_depth=6,
                              random_state=SEED, verbose=-1),
    'XGBoost': XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6,
                            random_state=SEED, verbosity=0)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'RMSE': root_mean_squared_error(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds)
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.2f}, MAE={results[name]["MAE"]:.2f}, R²={results[name]["R2"]:.4f}')

## 22 · Top 3 Model Selection

In [ ]:
all_results = {}
all_results['Baseline_LR'] = {'RMSE': baseline_rmse, 'MAE': baseline_mae, 'R2': baseline_r2}
if flaml_r2 is not None:
    all_results['FLAML'] = {'RMSE': flaml_rmse, 'MAE': flaml_mae, 'R2': flaml_r2}
all_results.update(results)

ranked = sorted(all_results.items(), key=lambda x: x[1]['RMSE'])
print('All models ranked by RMSE:')
for name, m in ranked:
    print(f"  {name:20s}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  R²={m['R2']:.4f}")

top3_names = [r[0] for r in ranked[:3]]
print(f'\nTop 3: {top3_names}')

## 23 · Final Training and Evaluation of Top 3

In [ ]:
print('Top 3 Final Results:')
print('=' * 60)
for name in top3_names:
    m = all_results[name]
    print(f"{name}:")
    print(f"  RMSE: {m['RMSE']:.2f}")
    print(f"  MAE:  {m['MAE']:.2f}")
    print(f"  R²:   {m['R2']:.4f}")
    print()

## 24 · Error Analysis

In [ ]:
best_name = top3_names[0]
if best_name in models:
    best_model = models[best_name]
else:
    best_model = models['CatBoost']

y_pred_best = best_model.predict(X_test)
residuals = y_test.values - y_pred_best

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_pred_best, residuals, alpha=0.5)
axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Predicted')

axes[1].hist(residuals, bins=30, edgecolor='black')
axes[1].set_title('Residual Distribution')

axes[2].scatter(y_test, y_pred_best, alpha=0.5)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted')
axes[2].set_title('Actual vs Predicted')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100)
plt.show()

## 25 · Interpretation and Business Insight

- **Job Title** and **Sector/Industry** are the strongest salary predictors.
- **Location (state)** significantly affects compensation — Silicon Valley commands premiums.
- **Company Rating** has a modest positive correlation with salary.
- These insights help job seekers target high-paying sectors and locations.

## 26 · Limitations

- Only 956 job postings — small sample.
- Salary Estimate parsing loses precision.
- Point-in-time snapshot — salaries change rapidly.
- Glassdoor bias: self-reported and estimated data.

## 27 · How to Improve This Project

1. NLP on Job Description for skill-based features.
2. Add cost-of-living adjustment by location.
3. Collect more recent data.
4. Include experience level features.
5. Use NER to extract required skills.

## 28 · Production Considerations

- Need real-time job market data.
- Regular retraining as market evolves.
- Privacy: individual salary data is sensitive.
- Confidence intervals for negotiations.

## 29 · Common Mistakes

1. Using raw Salary Estimate string in models.
2. Not handling -1 as missing in Rating/Founded.
3. Including Job Description text directly (too high-dimensional).
4. Not extracting state from Location.

## 30 · Mini Challenge / Exercises

1. Parse min and max salary separately and predict both.
2. Add TF-IDF features from Job Description.
3. Compare salary predictions for DS vs. ML Engineer vs. Data Analyst.
4. Adjust predictions by state-level cost of living.

## 31 · Final Summary / Key Takeaways

- Real-world job data requires significant cleaning (salary parsing, missing values).
- Job title, industry, and location drive salary predictions.
- With 956 rows, simpler models can compete with complex ensembles.
- Salary prediction models help both job seekers and recruiters.

In [ ]:
metrics = {}
for name in top3_names:
    metrics[name] = all_results[name]
metrics['baseline'] = all_results['Baseline_LR']

with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to metrics.json')
print('\nNotebook complete.')